# Oregon 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Oregon, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals).

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `dem_primary_total`, `rep_general_total`, `dem_general_total`, `lib_general_total`, `cst_general_total`, `grn_general_total`, `pro_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [3]:
import re 
import pandas as pd
import numpy as numpy
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

In [4]:
# OR 2008 dataset path
PRIMARY_PATH = r"../../data/raw/2008/OR/20080520__or__primary.csv"
GENERAL_PATH = r"../../data/raw/2008/OR/20081104__or__general.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/OR/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [5]:
# Load primary data
primary_df = pd.read_csv(PRIMARY_PATH)
primary_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Baker,President,NaN,D,Hillary Clinton,1089
1,Benton,President,NaN,D,Hillary Clinton,5530
2,Clackamas,President,NaN,D,Hillary Clinton,28149
3,Clatsop,President,NaN,D,Hillary Clinton,3266
4,Columbia,President,NaN,D,Hillary Clinton,4789
5,Coos,President,NaN,D,Hillary Clinton,5732
6,Crook,President,NaN,D,Hillary Clinton,1389
7,Curry,President,NaN,D,Hillary Clinton,1657
8,Deschutes,President,NaN,D,Hillary Clinton,8479
9,Douglas,President,NaN,D,Hillary Clinton,7730


In [6]:
# Different values in 'office' column
primary_df["office"].value_counts()

office
State House           526
U.S. Senate           360
Secretary of State    252
U.S. House            225
State Senate          223
President             213
Attorney General      180
State Treasurer       144
Name: count, dtype: int64

In [7]:
# Only keep rows where 'office' is 'President'
primary_df = primary_df[primary_df["office"] == "President"]
primary_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Baker,President,NaN,D,Hillary Clinton,1089
1,Benton,President,NaN,D,Hillary Clinton,5530
2,Clackamas,President,NaN,D,Hillary Clinton,28149
3,Clatsop,President,NaN,D,Hillary Clinton,3266
4,Columbia,President,NaN,D,Hillary Clinton,4789
5,Coos,President,NaN,D,Hillary Clinton,5732
6,Crook,President,NaN,D,Hillary Clinton,1389
7,Curry,President,NaN,D,Hillary Clinton,1657
8,Deschutes,President,NaN,D,Hillary Clinton,8479
9,Douglas,President,NaN,D,Hillary Clinton,7730


In [8]:
# Primary data shape when only considering President
primary_df.shape

(213, 6)

In [9]:
# Number of missing values in each column
primary_df.isna().sum()

county         0
office         0
district     213
party          0
candidate      0
votes          0
dtype: int64

In [10]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the district column since it's all missing values
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Baker,D,Hillary Clinton,1089
1,Benton,D,Hillary Clinton,5530
2,Clackamas,D,Hillary Clinton,28149
3,Clatsop,D,Hillary Clinton,3266
4,Columbia,D,Hillary Clinton,4789
5,Coos,D,Hillary Clinton,5732
6,Crook,D,Hillary Clinton,1389
7,Curry,D,Hillary Clinton,1657
8,Deschutes,D,Hillary Clinton,8479
9,Douglas,D,Hillary Clinton,7730


In [11]:
# Different candidates in primary_df
primary_df["candidate"].value_counts()

candidate
Misc.              71
John McCain        36
Ron Paul           36
Hillary Clinton    35
Barack Obama       35
Name: count, dtype: int64

In [12]:
# List out all the parties in the general election data
primary_df["party"].value_counts()

party
R    108
D    105
Name: count, dtype: int64

Notice that there is a "Misc." value in `candidate`. Yet, each of these miscellaneous values have an associated party (either R or D) with each. Then, we will indeed keep these misc. value to later count the total votes by party.

In [13]:
# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Baker,D,Hillary Clinton,1089
1,Benton,D,Hillary Clinton,5530
2,Clackamas,D,Hillary Clinton,28149
3,Clatsop,D,Hillary Clinton,3266
4,Columbia,D,Hillary Clinton,4789
5,Coos,D,Hillary Clinton,5732
6,Crook,D,Hillary Clinton,1389
7,Curry,D,Hillary Clinton,1657
8,Deschutes,D,Hillary Clinton,8479
9,Douglas,D,Hillary Clinton,7730


In [14]:
# Shape after preprocessing
primary_df.shape

(213, 4)

### b. General Election Dataset

In [24]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Baker,President,NaN,P,Ralph Nader,106
1,Benton,President,NaN,P,Ralph Nader,427
2,Clackamas,President,NaN,P,Ralph Nader,1750
3,Clatsop,President,NaN,P,Ralph Nader,249
4,Columbia,President,NaN,P,Ralph Nader,307
5,Coos,President,NaN,P,Ralph Nader,422
6,Crook,President,NaN,P,Ralph Nader,157
7,Curry,President,NaN,P,Ralph Nader,174
8,Deschutes,President,NaN,P,Ralph Nader,702
9,Douglas,President,NaN,P,Ralph Nader,561


In [25]:
# Different values in 'office' column
general_df["office"].value_counts()

office
State House           328
President             252
U.S. House            215
Attorney General      180
U.S. Senate           144
Secretary of State    144
State Treasurer       144
State Senate          131
Name: count, dtype: int64

In [26]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Baker,President,NaN,P,Ralph Nader,106
1,Benton,President,NaN,P,Ralph Nader,427
2,Clackamas,President,NaN,P,Ralph Nader,1750
3,Clatsop,President,NaN,P,Ralph Nader,249
4,Columbia,President,NaN,P,Ralph Nader,307
5,Coos,President,NaN,P,Ralph Nader,422
6,Crook,President,NaN,P,Ralph Nader,157
7,Curry,President,NaN,P,Ralph Nader,174
8,Deschutes,President,NaN,P,Ralph Nader,702
9,Douglas,President,NaN,P,Ralph Nader,561


In [27]:
# General data shape when only considering President
general_df.shape

(252, 6)

In [28]:
# Number of missing values in each column
general_df.isna().sum()

county         0
office         0
district     252
party         36
candidate      0
votes          0
dtype: int64

In [29]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Baker,P,Ralph Nader,106
1,Benton,P,Ralph Nader,427
2,Clackamas,P,Ralph Nader,1750
3,Clatsop,P,Ralph Nader,249
4,Columbia,P,Ralph Nader,307
5,Coos,P,Ralph Nader,422
6,Crook,P,Ralph Nader,157
7,Curry,P,Ralph Nader,174
8,Deschutes,P,Ralph Nader,702
9,Douglas,P,Ralph Nader,561


There are values in `party` that we need to further looking into those.

In [32]:
# Observation with missing value in `party`
general_df.loc[general_df["party"].isna()]

,county,party,candidate,votes
216,Baker,NaN,Misc.,87
217,Benton,NaN,Misc.,334
218,Clackamas,NaN,Misc.,1364
219,Clatsop,NaN,Misc.,167
220,Columbia,NaN,Misc.,259
221,Coos,NaN,Misc.,304
222,Crook,NaN,Misc.,76
223,Curry,NaN,Misc.,116
224,Deschutes,NaN,Misc.,504
225,Douglas,NaN,Misc.,494


In [34]:
# Different values in "`candidate` of `party` missing observations
general_df.loc[general_df["party"].isna()]["candidate"].value_counts()

candidate
Misc.    36
Name: count, dtype: int64

Indeed, all of the rows with missing values in `party` column have "Misc." `candidate`. We can just drop these rows without affecting anything with our current or future use.

In [40]:
# Drop observation with misisng value in `party`
general_df = general_df[
    general_df["party"].notna()
].reset_index(drop=True)

# Shape of general_df after dropping such observation
general_df.shape

(216, 4)

In [41]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
Ralph Nader         36
Cynthia McKinney    36
John McCain         36
Bob Barr            36
Chuck Baldwin       36
Barack Obama        36
Name: count, dtype: int64

In [42]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
P     36
PG    36
R     36
L     36
C     36
D     36
Name: count, dtype: int64

In [43]:
# Data type of each column in general_df
general_df.dtypes

county       object
party        object
candidate    object
votes         int64
dtype: object

In [44]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Baker,P,Ralph Nader,106
1,Benton,P,Ralph Nader,427
2,Clackamas,P,Ralph Nader,1750
3,Clatsop,P,Ralph Nader,249
4,Columbia,P,Ralph Nader,307
5,Coos,P,Ralph Nader,422
6,Crook,P,Ralph Nader,157
7,Curry,P,Ralph Nader,174
8,Deschutes,P,Ralph Nader,702
9,Douglas,P,Ralph Nader,561


In [45]:
# Shape after preprocessing
general_df.shape

(216, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [52]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "D"  : "dem",                      
                "R"  : "rep",
                "L"  : "lib",
                "C"  : "cst",
                "P"  : "pro",
                "Pg" : "grn" 
               })
           .fillna(s.str.strip().str.lower()))      # For all others, they're all three-letter abbr, just need to lowercase

In [53]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [54]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [55]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_dem_CLINTON,pri_dem_MISC,pri_dem_OBAMA,pri_rep_MCCAIN,pri_rep_MISC,pri_rep_PAUL
0,Baker,1089,84,1117,2229,115,446
1,Benton,5530,111,12647,6007,337,1155
2,Clackamas,28149,443,33606,30211,1749,5233
3,Clatsop,3266,74,3809,2722,177,480
4,Columbia,4789,124,4324,3395,190,759
5,Coos,5732,273,4925,5584,400,1370
6,Crook,1389,62,1243,2624,141,325
7,Curry,1657,88,1822,2778,176,628
8,Deschutes,8479,239,13441,15254,706,2031
9,Douglas,7730,329,7331,13558,703,2798


Note that there is a column with miscellaneous candidate that still had votes (`pri_dem_MISC`). We will keep this for total counting purposes and drop it at the end.

In [56]:
# Primary dataframe shape after pivot
primary_pivot.shape

(36, 7)

In [57]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_pro_NADER,gen_rep_MCCAIN
0,Baker,55,2805,23,51,106,5650
1,Benton,169,29901,169,214,427,15264
2,Clackamas,692,103476,284,717,1750,83595
3,Clatsop,70,10701,68,101,249,7192
4,Columbia,202,13390,74,123,307,10413
5,Coos,204,14401,103,163,422,15354
6,Crook,37,3632,24,55,157,6371
7,Curry,83,5230,26,57,174,6646
8,Deschutes,259,38819,129,305,702,39064
9,Douglas,320,20298,128,217,561,30919


In [58]:
# General dataframe shape after pivot
general_pivot.shape

(36, 7)

## 4. Merge Dataframes

Before merging, we verify that county names match across primary and general:

In [59]:
# Check if county names match between primary_df and general_df
primary_counties = set(primary_pivot["county"].unique())
general_counties = set(general_pivot["county"].unique())
common_counties = primary_counties.intersection(general_counties)
print(f"Number of common counties: {len(common_counties)} out of {len(primary_counties)}")

Number of common counties: 36 out of 36


Great. Since we know that all counties name are matched, we don't need to perform further data preprocessing to match the county names. Thus, we can now merge them:

In [60]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_CLINTON,pri_dem_MISC,pri_dem_OBAMA,pri_rep_MCCAIN,pri_rep_MISC,pri_rep_PAUL,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_pro_NADER,gen_rep_MCCAIN
0,Baker,1089,84,1117,2229,115,446,55,2805,23,51,106,5650
1,Benton,5530,111,12647,6007,337,1155,169,29901,169,214,427,15264
2,Clackamas,28149,443,33606,30211,1749,5233,692,103476,284,717,1750,83595
3,Clatsop,3266,74,3809,2722,177,480,70,10701,68,101,249,7192
4,Columbia,4789,124,4324,3395,190,759,202,13390,74,123,307,10413
5,Coos,5732,273,4925,5584,400,1370,204,14401,103,163,422,15354
6,Crook,1389,62,1243,2624,141,325,37,3632,24,55,157,6371
7,Curry,1657,88,1822,2778,176,628,83,5230,26,57,174,6646
8,Deschutes,8479,239,13441,15254,706,2031,259,38819,129,305,702,39064
9,Douglas,7730,329,7331,13558,703,2798,320,20298,128,217,561,30919


In [61]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_dem_CLINTON,pri_dem_MISC,pri_dem_OBAMA,pri_rep_MCCAIN,pri_rep_MISC,pri_rep_PAUL,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_pro_NADER,gen_rep_MCCAIN
count,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000
mean,7054.361111,170.277778,10241.527778,7941.138889,458.194444,1419.444444,213.694444,28813.638889,126.194444,212.083333,517.055556,20513.194444
std,11743.959447,198.284876,20802.913087,9410.399045,604.664338,1628.293839,249.797726,54451.243686,220.629984,300.003179,806.203021,24863.645751
min,0.000000,0.000000,0.000000,214.000000,12.000000,39.000000,5.000000,281.000000,1.000000,2.000000,8.000000,498.000000
25%,872.500000,46.500000,778.000000,1798.750000,72.750000,309.500000,38.500000,2913.000000,21.000000,30.500000,96.250000,4117.750000
50%,2836.500000,86.000000,2540.500000,3315.000000,183.500000,650.500000,84.000000,9427.000000,62.000000,107.000000,247.000000,8186.000000
75%,6177.500000,240.500000,7336.250000,8944.000000,509.750000,1618.500000,266.000000,21138.500000,130.250000,221.250000,509.250000,24247.500000
max,56945.000000,882.000000,110280.000000,32786.000000,2225.000000,5467.000000,904.000000,279696.000000,1207.000000,1195.000000,4166.000000,89185.000000


Now, we will add party totals columns: 

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    * `dem_primary_total` = sum of all `pri_dem_*` columns

- General totals:
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `lib_general_total` = sum of all `gen_lib_*` columns
    * `cst_general_total` = sum of all `gen_cst_*` columns
    * `grn_general_total` = sum of all `gen_grn_*` columns
    * `pro_general_total` = sum of all `gen_pro_*` columns

In [62]:
# Add party totals for primary election
rep_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_dem_")]

merged_df["rep_primary_total"] = merged_df[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
merged_df["dem_primary_total"] = merged_df[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0

Now, we have calculated the total vote for each party. Thus, we can drop the miscellaneous column.

In [63]:
# Drop UNCOMMITTED column for primary election
merged_df = merged_df.drop(columns="pri_dem_MISC")

# Snippet at the merged dataframe with primary totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_CLINTON,pri_dem_OBAMA,pri_rep_MCCAIN,pri_rep_MISC,pri_rep_PAUL,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_pro_NADER,gen_rep_MCCAIN,rep_primary_total,dem_primary_total
0,Baker,1089,1117,2229,115,446,55,2805,23,51,106,5650,2790,2290
1,Benton,5530,12647,6007,337,1155,169,29901,169,214,427,15264,7499,18288
2,Clackamas,28149,33606,30211,1749,5233,692,103476,284,717,1750,83595,37193,62198
3,Clatsop,3266,3809,2722,177,480,70,10701,68,101,249,7192,3379,7149
4,Columbia,4789,4324,3395,190,759,202,13390,74,123,307,10413,4344,9237
5,Coos,5732,4925,5584,400,1370,204,14401,103,163,422,15354,7354,10930
6,Crook,1389,1243,2624,141,325,37,3632,24,55,157,6371,3090,2694
7,Curry,1657,1822,2778,176,628,83,5230,26,57,174,6646,3582,3567
8,Deschutes,8479,13441,15254,706,2031,259,38819,129,305,702,39064,17991,22159
9,Douglas,7730,7331,13558,703,2798,320,20298,128,217,561,30919,17059,15390


In [64]:
# Add party totals for general election
rep_general_cols   = [c for c in merged_df.columns if c.startswith("gen_rep_")]
dem_general_cols   = [c for c in merged_df.columns if c.startswith("gen_dem_")]
lib_general_cols   = [c for c in merged_df.columns if c.startswith("gen_lib_")]
cst_general_cols   = [c for c in merged_df.columns if c.startswith("gen_cst_")]
grn_general_cols   = [c for c in merged_df.columns if c.startswith("gen_grn_")]
pro_general_cols   = [c for c in merged_df.columns if c.startswith("gen_pro_")]

merged_df["rep_general_total"] = merged_df[rep_general_cols].sum(axis=1) if rep_general_cols else 0
merged_df["dem_general_total"] = merged_df[dem_general_cols].sum(axis=1) if dem_general_cols else 0
merged_df["lib_general_total"] = merged_df[lib_general_cols].sum(axis=1) if lib_general_cols else 0
merged_df["cst_general_total"] = merged_df[cst_general_cols].sum(axis=1) if cst_general_cols else 0
merged_df["grn_general_total"] = merged_df[grn_general_cols].sum(axis=1) if grn_general_cols else 0
merged_df["pro_general_total"] = merged_df[pro_general_cols].sum(axis=1) if pro_general_cols else 0

In [65]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned dataframe:")
merged_df.columns

Final columns in the cleaned dataframe:


Index(['county', 'pri_dem_CLINTON', 'pri_dem_OBAMA', 'pri_rep_MCCAIN',
       'pri_rep_MISC', 'pri_rep_PAUL', 'gen_cst_BALDWIN', 'gen_dem_OBAMA',
       'gen_grn_MCKINNEY', 'gen_lib_BARR', 'gen_pro_NADER', 'gen_rep_MCCAIN',
       'rep_primary_total', 'dem_primary_total', 'rep_general_total',
       'dem_general_total', 'lib_general_total', 'cst_general_total',
       'grn_general_total', 'pro_general_total'],
      dtype='object')

In [66]:
# Preview merged dataframe with totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_CLINTON,pri_dem_OBAMA,pri_rep_MCCAIN,pri_rep_MISC,pri_rep_PAUL,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_pro_NADER,gen_rep_MCCAIN,rep_primary_total,dem_primary_total,rep_general_total,dem_general_total,lib_general_total,cst_general_total,grn_general_total,pro_general_total
0,Baker,1089,1117,2229,115,446,55,2805,23,51,106,5650,2790,2290,5650,2805,51,55,23,106
1,Benton,5530,12647,6007,337,1155,169,29901,169,214,427,15264,7499,18288,15264,29901,214,169,169,427
2,Clackamas,28149,33606,30211,1749,5233,692,103476,284,717,1750,83595,37193,62198,83595,103476,717,692,284,1750
3,Clatsop,3266,3809,2722,177,480,70,10701,68,101,249,7192,3379,7149,7192,10701,101,70,68,249
4,Columbia,4789,4324,3395,190,759,202,13390,74,123,307,10413,4344,9237,10413,13390,123,202,74,307
5,Coos,5732,4925,5584,400,1370,204,14401,103,163,422,15354,7354,10930,15354,14401,163,204,103,422
6,Crook,1389,1243,2624,141,325,37,3632,24,55,157,6371,3090,2694,6371,3632,55,37,24,157
7,Curry,1657,1822,2778,176,628,83,5230,26,57,174,6646,3582,3567,6646,5230,57,83,26,174
8,Deschutes,8479,13441,15254,706,2031,259,38819,129,305,702,39064,17991,22159,39064,38819,305,259,129,702
9,Douglas,7730,7331,13558,703,2798,320,20298,128,217,561,30919,17059,15390,30919,20298,217,320,128,561


Now, we save the cleaned dataframe into the processed directory.

In [67]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "OR.csv", index=False)